# Pydantic AI — typed agents with structured output

Pydantic AI is what the OpenAI Python client would look like if you wrote it after building Pydantic. The point: **types**.

* `Agent[Deps, Output]` — `Deps` is the typed bag of resources tools can read; `Output` is the typed shape the agent returns.
* `@agent.tool` — Python type hints become the JSON schema. No boilerplate.
* Set `output_type=PaperSummary` — you get a Pydantic model back, not a string.

## Canonical code (with `pydantic-ai` installed)

```python
from dataclasses import dataclass
from pydantic import BaseModel
from pydantic_ai import Agent, RunContext
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from task import search_corpus, fetch_paper, cite

@dataclass
class Deps:
    user_id: str

class PaperSummary(BaseModel):
    arxiv_id: str
    headline: str
    method: str
    metric: str

agent = Agent('openai:gpt-4o-mini', deps_type=Deps, output_type=PaperSummary)

@agent.tool
async def search(ctx: RunContext[Deps], query: str) -> list[dict]:
    return search_corpus(query, k=2)

@agent.tool
async def fetch(ctx: RunContext[Deps], arxiv_id: str) -> dict:
    return fetch_paper(arxiv_id)

result = await agent.run('What is RA-MoE?', deps=Deps(user_id='demo'))
print(result.output)  # PaperSummary instance, NOT a string
```

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / '03-agentic-frameworks'))
if not (os.getenv('OPENAI_API_KEY') or os.getenv('ANTHROPIC_API_KEY')):
    os.environ.setdefault('LLM_CACHE_ONLY', '1')
from task import get_question
from eval import pydantic_ai_solve
trace = pydantic_ai_solve(get_question('q01'))
for step in trace.steps:
    print(f"{step.role}: {step.name or ''}  {step.output_summary or step.content or ''}"[:140])

## Tradeoffs

* **+ Types end-to-end** — no string-parsing your agent's output.
* **+ JSON-schema-from-type-hints** — tool definitions are minimal.
* **− Minimal graph primitives** — for branchy multi-agent workflows, LangGraph or CrewAI fit better.
* **− Newer ecosystem** — fewer cookbooks than LangChain/LangGraph (improving fast).